In [16]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [17]:
ruta_pipeline = "../models/rotten_pipeline.pkl"
with open(ruta_pipeline, "rb") as f:
    pipeline_cargado = pickle.load(f)

In [18]:
##movies = pd.read_csv("../data/processed/movies.csv")
movies = pd.read_csv("dataset/movies.csv")

In [29]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17712 entries, 0 to 17711
Data columns (total 23 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   rotten_tomatoes_link              17712 non-null  object 
 1   movie_title                       17712 non-null  object 
 2   movie_info                        17391 non-null  object 
 3   critics_consensus                 9134 non-null   object 
 4   content_rating                    17712 non-null  object 
 5   genres                            17712 non-null  object 
 6   runtime                           17398 non-null  float64
 7   production_company                17213 non-null  object 
 8   tomatometer_status                17668 non-null  object 
 9   tomatometer_rating                17668 non-null  float64
 10  tomatometer_count                 17668 non-null  float64
 11  audience_status                   17264 non-null  object 
 12  audi

In [20]:
##Estos son ejemplos, pueden estar abajo
selected_genres = ["Action & Adventure", "Fantasy"]
selected_actor = "Logan Lerman"
selected_director = "Chris Columbus"
selected_year_range = (2005, 2015)
selected_tomatometer = 50
selected_movie = "Harry Potter and the Goblet of Fire"
top_n = 5
alpha = 0.7   # peso a la pelicula elegida (el titulo)
beta = 0.2    # peso al resto de valores de imput
gamma = 0.1   # peso sentimiento del crítico


In [21]:
selected_genres = ["Action & Adventure", "Animation"]
selected_actor = ""
selected_director = "Zack Snyder"
selected_year_range = ""
selected_tomatometer = 50
selected_movie = "300"
top_n = 5
alpha = 0.7   # peso a la pelicula elegida (el titulo)
beta = 0.2    # peso al resto de valores de imput
gamma = 0.1   # peso sentimiento del crítico


In [22]:
# Preparar dataset
filtered_movies = movies.copy()

# Se rellenan los nulos para que no de error
for col in ["genres", "actors", "directors", "movie_info", "critics_consensus"]:
    filtered_movies[col] = filtered_movies[col].fillna("")
filtered_movies["release_year"] = filtered_movies["release_year"].fillna("1900")
filtered_movies["tomatometer_rating"] = filtered_movies["tomatometer_rating"].fillna(0)

In [23]:
# Aqui se crea la coincidencia que tiene con los imput puestos y se da un puntaje con cada coincidencia
filtered_movies["match_score"] = 0
if selected_genres:
    filtered_movies["match_score"] += filtered_movies["genres"].apply(lambda x: sum(g in x for g in selected_genres))
if selected_actor:
    filtered_movies["match_score"] += filtered_movies["actors"].apply(lambda x: 2 if selected_actor in x else 0)
if selected_director:
    filtered_movies["match_score"] += filtered_movies["directors"].apply(lambda x: 2 if selected_director in x else 0)
if selected_year_range:
    filtered_movies["match_score"] += filtered_movies["original_release_date"].apply(
        lambda x: 1 if selected_year_range[0] <= int(str(x)[:4]) <= selected_year_range[1] else 0
    )
if selected_tomatometer is not None:
    filtered_movies["match_score"] += filtered_movies["tomatometer_rating"].apply(lambda x: 1 if x >= selected_tomatometer else 0)

In [24]:
# Normalizar match_score que sirve para los calculos de mas abajo
filtered_movies["match_norm"] = filtered_movies["match_score"] / filtered_movies["match_score"].max()

# Aplicar modelo de sentimiento a critics_consensus (recuerden que la importancia la controla gamma)
filtered_movies["consensus_sentiment_prob"] = pipeline_cargado.predict_proba(
    filtered_movies["critics_consensus"]
)[:, 1]
filtered_movies["consensus_sentiment_norm"] = filtered_movies["consensus_sentiment_prob"] / filtered_movies["consensus_sentiment_prob"].max()

# Resetear índices para poder usar iloc después
filtered_movies = filtered_movies.reset_index(drop=True)

# TF-IDF sobre combined_features que reune en un solo campo los textos 
filtered_movies["combined_features"] = (
    filtered_movies["genres"] + " " +
    filtered_movies["directors"] + " " +
    filtered_movies["actors"] + " " +
    filtered_movies["movie_info"]
)
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(filtered_movies["combined_features"])

In [25]:
# Función recomendación
def recomendar_por_similitud(selected_movie=None, top_n=5, alpha=0.7, beta=0.2, gamma=0.1):
    # caso de que no haya películas que cumplan los filtros o no haya info de referencia
    if filtered_movies.empty:
        print("No hay películas que cumplan los filtros.")
        return pd.DataFrame(columns=["movie_title", "genres", "actors", "directors", "match_score"])
    
    # Si no hay película de referencia o no está en el dataset
    if selected_movie is None or selected_movie not in filtered_movies["movie_title"].values:

        # Si no hay película de referencia, solo usar match_score + sentimiento (por si el camo de titulo esta vacio o no existe)
        print("⚠️ No se proporcionó película de referencia o no está en el dataset. Usando match_score + sentimiento.")

        # se usa solo match_score y sentimiento (es importante mostrar el print por que puede que los resultados no tengan que ver y sea por que estaba mal escrita el nombre de la pelicula)
        final_score = beta * filtered_movies["match_norm"] + gamma * filtered_movies["consensus_sentiment_norm"]
        top_indices = final_score.argsort()[::-1][:top_n]
        return filtered_movies.iloc[top_indices][
            ["movie_title", "genres", "actors", "directors", "match_score", "consensus_sentiment_prob"]
        ]
    
    # Película de referencia existe
    idx = filtered_movies[filtered_movies["movie_title"] == selected_movie].index[0]
    # Similaridad coseno entre la película seleccionada y todas las demás
    cosine_sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    
    # Score combinado
    final_score = (
        alpha * cosine_sim +
        beta * filtered_movies["match_norm"] +
        gamma * filtered_movies["consensus_sentiment_norm"]
    )
    
    # Obtener los índices de las películas con mayor puntaje combinado
    top_indices = final_score.argsort()[::-1]
    top_indices = [i for i in top_indices if i != idx]  # excluir la misma película
    top_indices = top_indices[:top_n]
    top_indices = filtered_movies["match_score"].argsort()[::-1][:top_n] #Lo oredena por match score para que las primeras sean las que mas coincidan con los filtros que es lo mas importante

    
    # Devuelve las recomendaciones, si quieren mostrar los datos busquen los titulos en el dataset original y muestren lo que vean necesario
    return filtered_movies.iloc[top_indices][
        ["movie_title", "genres", "actors", "directors", "match_score", "consensus_sentiment_prob"]
    ]

In [26]:
#Aqui se aplican los ejemplos
recommendations = recomendar_por_similitud(selected_movie, top_n=top_n, alpha=alpha, beta=beta, gamma=gamma)
recommendations

,movie_title,genres,actors,directors,match_score,consensus_sentiment_prob
9376,Legend of the Guardians: The Owls of Ga'Hoole,"['Action & Adventure', 'Animation', 'Science F...","['Abbie Cornish', 'Miriam Margolyes', 'Helen M...",['Zack Snyder'],5,0.618346
17066,Watchmen,"['Action & Adventure', 'Drama', 'Science Ficti...","['Billy Crudup', 'Jeffrey Dean Morgan', 'Malin...",['Zack Snyder'],4,0.866974
1956,300,"['Action & Adventure', 'Drama']","['Gerard Butler', 'Lena Headey', 'David Wenham...",['Zack Snyder'],4,0.679015
14095,Man of Steel,"['Action & Adventure', 'Science Fiction & Fant...","['Henry Cavill', 'Amy Adams', 'Michael Shannon...",['Zack Snyder'],4,0.795595
7993,How to Train Your Dragon 2,"['Action & Adventure', 'Animation', 'Kids & Fa...","['Jay Baruchel', 'Gerard Butler', 'America Fer...",['Dean DeBlois'],3,0.988240


In [27]:
import requests
api_key = "fa1b7162" ##Esto es lo importante, no lo borren ni cambien
peliculas = recommendations["movie_title"].tolist()
peliculas = ["Toy Story"]
datos_peliculas = []

for titulo in peliculas:
    url = f"http://www.omdbapi.com/?t={titulo}&apikey={api_key}"
    response = requests.get(url)
    data = response.json()
    if data["Response"] == "True":
        datos_peliculas.append(data)

datos_peliculas

[{'Title': 'Toy Story',
  'Year': '1995',
  'Rated': 'G',
  'Released': '22 Nov 1995',
  'Runtime': '81 min',
  'Genre': 'Animation, Adventure, Comedy',
  'Director': 'John Lasseter',
  'Writer': 'Joss Whedon, Andrew Stanton, Joel Cohen',
  'Actors': 'Tom Hanks, Tim Allen, Don Rickles',
  'Plot': "A cowboy doll is profoundly jealous when a new spaceman action figure supplants him as the top toy in a boy's bedroom. When circumstances separate them from their owner, the duo have to put aside their differences to return to him.",
  'Language': 'English',
  'Country': 'United States',
  'Awards': 'Nominated for 3 Oscars. 29 wins & 24 nominations total',
  'Poster': 'https://m.media-amazon.com/images/M/MV5BZTA3OWVjOWItNjE1NS00NzZiLWE1MjgtZDZhMWI1ZTlkNzYwXkEyXkFqcGc@._V1_SX300.jpg',
  'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.3/10'},
   {'Source': 'Rotten Tomatoes', 'Value': '100%'},
   {'Source': 'Metacritic', 'Value': '96/100'}],
  'Metascore': '96',
  'imdbRating': '8.

In [28]:
movies.head()

,rotten_tomatoes_link,movie_title,movie_info,critics_consensus,content_rating,genres,runtime,production_company,tomatometer_status,tomatometer_rating,...,audience_count,tomatometer_top_critics_count,tomatometer_fresh_critics_count,tomatometer_rotten_critics_count,directors,authors,actors,release_year,streaming_release_year,avg_sentiment_score
0,m/0814255,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",Though it may seem like just another Harry Pot...,PG,"['Action & Adventure', 'Comedy', 'Drama', 'Sci...",119.0,20th Century Fox,Rotten,49.0,...,254421.0,43,73,76,['Chris Columbus'],"['Craig Titley', 'Chris Columbus', 'Rick Riord...","['Logan Lerman', 'Brandon T. Jackson', 'Alexan...",2010.0,2015,0.555641
1,m/0878835,Please Give,Kate (Catherine Keener) and her husband Alex (...,Nicole Holofcener's newest might seem slight i...,R,['Comedy'],90.0,Sony Pictures Classics,Certified-Fresh,87.0,...,11574.0,44,123,19,['Nicole Holofcener'],['Nicole Holofcener'],"['Catherine Keener', 'Amanda Peet', 'Oliver Pl...",2010.0,2012,0.811973
2,m/10,10,"A successful, middle-aged Hollywood songwriter...",Blake Edwards' bawdy comedy may not score a pe...,R,"['Comedy', 'Romance']",122.0,Waner Bros.,Fresh,67.0,...,14684.0,2,16,8,['Blake Edwards'],['Blake Edwards'],"['Dudley Moore', 'Bo Derek', 'Julie Andrews', ...",1979.0,2014,0.587963
3,m/1000013-12_angry_men,12 Angry Men (Twelve Angry Men),Following the closing arguments in a murder tr...,Sidney Lumet's feature debut is a superbly wri...,NR,"['Classics', 'Drama']",95.0,Criterion Collection,Certified-Fresh,100.0,...,105386.0,6,54,0,['Sidney Lumet'],['Reginald Rose'],"['Martin Balsam', 'John Fiedler', 'Lee J. Cobb...",1957.0,2017,0.808762
4,m/1000079-20000_leagues_under_the_sea,"20,000 Leagues Under The Sea","In 1866, Professor Pierre M. Aronnax (Paul Luk...","One of Disney's finest live-action adventures,...",G,"['Action & Adventure', 'Drama', 'Kids & Family']",127.0,Disney,Fresh,89.0,...,68918.0,5,24,3,['Richard Fleischer'],['Earl Felton'],"['James Mason', 'Kirk Douglas', 'Paul Lukas', ...",1954.0,2016,0.768692
